In [ ]:
import sys
from pathlib import Path

sys.path.append('/Data_large/marine/PythonProjects/MMDET/notebooks/Tools/EigenCAM-Pytorch')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs/custom_components')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')

import torch
from cam import EigenCAM
from mmdet.datasets.transforms import Resize, RandomFlip, Pad, ImageToTensor
from mmdet.apis import init_detector, inference_detector
import mmcv

DATA_PATH_VEN = '/Data_large/marine/Datasets/VENuS/ds_L0/perfect'
DATA_PATH_SEN = '/Data_large/marine/Datasets/VDS2Raw/imgs'

TIFF_VEN = list(Path(DATA_PATH_VEN).rglob('*.tif'))
TIFF_SEN = list(Path(DATA_PATH_SEN).rglob('*.tif'))



# Specify the path to model config and checkpoint file
config_file = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b5/42_BS_3_LR_0.0009_ME_30_OPT_SGD/vfnet_r18.py'
checkpoint_file = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b5/42_BS_3_LR_0.0009_ME_30_OPT_SGD/epoch_30.pth'

# build the model from a config file and a checkpoint file
DetModel = init_detector(config_file, checkpoint_file, device='cuda:0')

In [ ]:
from LoaderUnreg import SelBandLoader
from PreProcessorUnreg import MyPrePro
from mmdet.models.data_preprocessors import DetDataPreprocessor


BAND = 5

########## TRANSFORMS ##########
loader = SelBandLoader(
            to_float32 = False,
            bands_list = [BAND],
            ignore_empty = False,
            backend_args = None
            )

IMG_SIZE = 2048
resizer = Resize(scale=(IMG_SIZE, IMG_SIZE), keep_ratio=False)
ToTensor = ImageToTensor(keys=['img'])

MEANS=[158.69588,124.42161,109.27108,105.380424,88.40926,98.93067,88.819916,94.20678,103.540764,111.64337,122.92817,79.31501]
STD=[34.95446,46.282494,56.252197,55.741932,64.54027,59.59095,69.65824,68.40028,77.930405,103.4634,105.30468,65.8369]
M, S = MEANS[BAND-1], STD[BAND-1]


prepro = MyPrePro(mean = [M], pad_size_divisor=1, std=[S])


################################
img_path = TIFF_VEN[0]
print(img_path)

IMG_k = loader.transform(results={'img_path': img_path})
IMG_k = resizer(IMG_k)
IMG_k = ToTensor(IMG_k)
tensor = IMG_k['img'].to('cuda:0').unsqueeze(0)
IMG_k = prepro({'inputs': tensor})

inputs = IMG_k['inputs']

Back-Testing encoder

In [ ]:
encoder = DetModel.backbone
out_features = encoder(inputs.to('cuda:0'))

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

layer_name = 'layer4'

cam_obj = EigenCAM(encoder, device, None, layer_name)

print('\ndevice:', device)
print('layer Name to plot heatmap:', cam_obj.layer_name)

# output is torch Tensor, overlay is ndarray
output, overlay = cam_obj.get_heatmap(inputs.to('cuda:0').squeeze(0))


In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=go.Image(z=overlay))
fig.update_layout(width=800, height=800, title='Overlay Image')
fig.show()
